In [3]:
### ============ 1. IMPORTS =====================
import pandas as pd # for data manipulation
import pickle # for saving the processed data to disk
import sys # for manipulating the Python path
import os # for file path manipulations
from scipy.io import arff # for loading ARFF files from OpenML
# !pip install ucimlrepo # for loading datasets from the UCI Machine Learning Repository
# from ucimlrepo import fetch_dataset # for fetching datasets from the UCI Machine Learning Repository

# Make the src/ package importable from within the notebooks/ subfolder.
sys.path.append("..")

# Pull in the same SEED and TEST_SIZE constants from config.py so that all notebooks use the same values.
from src.config import SEED, TEST_SIZE
from src.preprocess import split_data, scale_data

In [4]:
# # ===================== DATASET 1 - UCI Phishing Website Dataset (Full Variant) [16] =====
# Load the data file
data, meta = arff.loadarff('../data/raw/Phishing_Legitimate_full.arff')
# Convert to a Pandas data frame
df = pd.DataFrame(data)
# inspect the data
df.head()

,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,TildeSymbol,NumUnderscore,NumPercent,...,IframeOrFrame,MissingTitle,ImagesOnlyInForm,SubdomainLevelRT,UrlLengthRT,PctExtResourceUrlsRT,AbnormalExtFormActionR,ExtMetaScriptLinkRT,PctExtNullSelfRedirectHyperlinksRT,CLASS_LABEL
0,3.0,1.0,5.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,1.0,-1.0,1.0,b'1'
1,3.0,1.0,3.0,144.0,0.0,0.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,1.0,-1.0,1.0,1.0,1.0,1.0,b'1'
2,3.0,1.0,2.0,58.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,-1.0,1.0,-1.0,0.0,b'1'
3,3.0,1.0,6.0,79.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-1.0,1.0,1.0,1.0,-1.0,b'1'
4,3.0,0.0,4.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,-1.0,0.0,-1.0,-1.0,b'1'


In [9]:
# DATASET 3 - UCI Phishing Website Dataset (Small Variant) [18]
# # load the data file
# df = pd.read_csv('../data/raw/dataset_small.csv')
# # inspect the data
# print(df.shape)
# df.head()

In [10]:
# DATASET 4 - UCI Phishing Website Dataset (Full Variant) [19]
# load the data file
# df = pd.read_csv('../data/raw/dataset_full.csv')
# # inspect the data
# print(df.shape)
# df.head()

In [11]:
# ===== 3. DATA INSPECTION =====
# check the shape to confirm we have the expected number of rows and columns
# print(df.shape)
# # show summary statistics and data types to check for any obvious issues
# print(df.info()) 
# # check for missing values and duplicates
# print(df.isnull().sum())
# #  check if there are duplicates so they can be removed before training the GA
# print(df.duplicated().sum())

In [5]:
# ===== 4. PREPROCESSING =====
# Identify target and features
target_col = "CLASS_LABEL" # the name of the target variable column in the dataset

# Drop the label column to get the feature matrix X.
X = df.drop(columns=[target_col])

# convert bytes to int
y = df[target_col].astype(str).astype(int)

print("X shape:", X.shape[1])   # number of features
print("y shape:", y.shape)      # number of samples

# check the distribution of the target variable
print(y.value_counts())
print(y.unique())

X shape: 48
y shape: (10000,)
CLASS_LABEL
1    5000
0    5000
Name: count, dtype: int64
[1 0]


In [ ]:
# DATASET 2 - UCI Phishing Website Dataset [17]
# # load the data file
# data, meta = arff.loadarff('../data/raw/.old.arff')

# # Convert to a Pandas data frame
# df = pd.DataFrame(data)

# # convert bytes to int
# for col in df.select_dtypes(include=["object"]).columns:
#     df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

# # convert to int
# df = df.astype(int)

# # fix the target variable to be binary 0/1 instead of -1/1
# df["Result"] = df["Result"].replace({-1: 0, 1: 1})

# # inspect the data
# df.head()

# # split the data into features and target
# X = df.drop(columns=["Result"])
# y = df["Result"]

In [6]:
# ===== 6. TRAIN / TEST SPLIT =====
# Use the same SEED and TEST_SIZE constants from config.py to ensure all notebooks use the same split.
X_train, X_test, y_train, y_test = split_data(X, y, test_size=TEST_SIZE, seed=SEED)
print(X_train.shape, X_test.shape)  # confirm 70/30 ratio looks right

(7000, 48) (3000, 48)


In [7]:
# ===== 7. SCALE FEATURES =====
# scale the data using z-score normalization (standardization)
X_train_scaled, X_test_scaled, mean, std_replaced = scale_data(X_train, X_test)

# Verify the scaling worked by checking the mean and standard deviation of the scaled features.
print(X_train_scaled.describe().loc[["mean", "std"]])

           NumDots  SubdomainLevel     PathLevel     UrlLength       NumDash  \
mean -1.162245e-16   -5.075305e-17 -9.744586e-17  3.857232e-17  4.161750e-17   
std   1.000000e+00    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   

      NumDashInHostname      AtSymbol   TildeSymbol  NumUnderscore  \
mean      -7.562205e-17  1.522592e-18  4.060244e-18   2.740665e-17   
std        1.000000e+00  1.000000e+00  1.000000e+00   1.000000e+00   

        NumPercent  ...  SubmitInfoToEmail  IframeOrFrame  MissingTitle  \
mean  4.897670e-17  ...       2.842171e-17  -1.421085e-17  6.699403e-17   
std   1.000000e+00  ...       1.000000e+00   1.000000e+00  1.000000e+00   

      ImagesOnlyInForm  SubdomainLevelRT   UrlLengthRT  PctExtResourceUrlsRT  \
mean      5.684342e-17      1.502290e-16 -3.349701e-17          6.851662e-17   
std       1.000000e+00      1.000000e+00  1.000000e+00          1.000000e+00   

      AbnormalExtFormActionR  ExtMetaScriptLinkRT  \
mean           -4.567775e-17

In [8]:
# ===== 8. SAVE PROCESSED DATA =====
# Persist the split and scaled arrays to disk with pickle so that
# subsequent notebooks (02, 03, 04) can reload them without re-running
# the entire preprocessing pipeline from scratch.
with open("../data/processed/uci_plf/uci_plf_split_scaled.pkl", "wb") as f:
    pickle.dump((X_train_scaled, X_test_scaled, y_train, y_test), f)